# Section 4: Sarcasm Explanation & Error Analysis
**Author:** Shoeb Iqbal  

This notebook:
1. Retrains the best DeBERTa model (trained on ALL varieties, sarcasm task)
2. Runs inference on the combined test set to extract 10 erroneous predictions
3. Selects 4 explainable examples and builds a few-shot prompt
4. Tests the remaining 6 examples with that few-shot prompt
5. Comments on whether the few-shot prompt improved predictions

## 0. Setup

In [ ]:
!pip install datasets transformers torch scikit-learn -q
print("Libraries installed.")

In [ ]:
from huggingface_hub import login
login(token="your token here.")  # TODO: add your HuggingFace token
print("Logged in.")

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"GPU available: {torch.cuda.is_available()}")

---
## Step 1: Load Dataset (same as Section 2.2)

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

def get_variety_splits(variety):
    train = ds['train'].filter(lambda x: x['variety'] == variety)
    val   = ds['validation'].filter(lambda x: x['variety'] == variety)
    test  = ds['test'].filter(lambda x: x['variety'] == variety)
    return train, val, test

train_uk, val_uk, test_uk = get_variety_splits('en-UK')
train_in, val_in, test_in = get_variety_splits('en-IN')
train_au, val_au, test_au = get_variety_splits('en-AU')

train_combined = concatenate_datasets([train_uk, train_in, train_au])
val_combined   = concatenate_datasets([val_uk, val_in, val_au])
test_combined  = concatenate_datasets([test_uk, test_in, test_au])

print(f"Combined train: {len(train_combined)}")
print(f"Combined test:  {len(test_combined)}")

---
## Step 2: Retrain Best DeBERTa Model (ALL varieties, Sarcasm)

Replicating the exact setup from Section 2.2 — WeightedTrainer with class weights [1.0, 3.0] to handle sarcasm class imbalance.

In [ ]:
DEBERTA_MODEL = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def prepare_dataset(dataset, label_column):
    def add_labels(x):
        return {'labels': int(x[label_column])}
    dataset = dataset.map(add_labels)
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {'macro_f1': f1}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = torch.tensor([1.0, 3.0]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Setup complete.")

In [ ]:
train_ds = prepare_dataset(train_combined, 'Sarcasm')
val_ds   = prepare_dataset(val_combined,   'Sarcasm')

model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL,
    num_labels=2,
    problem_type="single_label_classification",
    torch_dtype=torch.float32,
)

args = TrainingArguments(
    output_dir='./results_deberta_ALL_sarcasm_seed1',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    seed=1,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete.")

---
## Step 3: Run Inference on Combined Test Set & Extract 10 Errors

In [ ]:
test_ds = prepare_dataset(test_combined, 'Sarcasm')
predictions = trainer.predict(test_ds)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Build results dataframe using text and variety from the raw dataset
test_texts     = test_combined['text']
test_varieties = test_combined['variety']

df_preds = pd.DataFrame({
    'text':            test_texts,
    'variety':         test_varieties,
    'true_label':      labels,
    'predicted_label': preds,
})

print(classification_report(
    labels, preds,
    target_names=['Not Sarcastic', 'Sarcastic']
))
print(f"Total test samples: {len(df_preds)}")

In [ ]:
# Extract all errors
df_errors = df_preds[df_preds['true_label'] != df_preds['predicted_label']].copy()
df_errors['error_type'] = df_errors.apply(
    lambda r: 'False Negative (missed sarcasm)' if r['true_label'] == 1
              else 'False Positive (wrongly flagged)',
    axis=1
)

print(f"Total errors: {len(df_errors)}")
print(df_errors['error_type'].value_counts())
print()
print(df_errors['variety'].value_counts())

In [ ]:
# Pick 10 diverse errors — prioritise false negatives (more interesting for sarcasm analysis)
# and ensure a mix of varieties
fn = df_errors[df_errors['error_type'] == 'False Negative (missed sarcasm)']
fp = df_errors[df_errors['error_type'] == 'False Positive (wrongly flagged)']

n_fn = min(7, len(fn))
n_fp = 10 - n_fn

df_10 = pd.concat([
    fn.sample(n_fn, random_state=42),
    fp.sample(min(n_fp, len(fp)), random_state=42)
]).reset_index(drop=True)

print(f"Selected {len(df_10)} errors:")
df_10[['text', 'variety', 'true_label', 'predicted_label', 'error_type']]

---
## Step 4: Choose 4 Examples & Write Explanations

Inspect `df_10` above and pick 4 examples you can clearly explain.

> ⚠️ **TODO:** Update `chosen_indices` and rewrite `explanations` to match your actual examples once you've seen the real errors.

In [ ]:
label_map = {1: 'Sarcastic', 0: 'Not Sarcastic'}

# TODO: Update these indices after inspecting df_10 above
chosen_indices = [0, 1, 2, 3]
df_4 = df_10.iloc[chosen_indices].copy().reset_index(drop=True)

# TODO: Replace with explanations that match your actual chosen examples
explanations = [
    "This text uses exaggerated positive language to describe a clearly frustrating situation. "
    "The contrast between the positive wording and the negative context is a strong sarcasm marker "
    "that DeBERTa missed, likely because the surface tokens appear positive.",

    "The sentence contains a rhetorical question with an implied obvious answer — a classic sarcasm device. "
    "DeBERTa missed the irony because without broader context the sentence reads as genuine inquiry.",

    "This is a false positive — the text appears sarcastic due to hyperbolic wording but is in fact sincere. "
    "The dialect-specific expression (en-AU) likely confused the model, as this phrasing carries genuine "
    "enthusiasm in Australian English.",

    "The sentence uses understatement to convey frustration — describing a major problem as merely "
    "'a bit inconvenient'. DeBERTa missed this because the individual words are mild and non-confrontational."
]

df_4['explanation'] = explanations

for i, row in df_4.iterrows():
    print(f"Example {i+1}:")
    print(f"  Text:        {row['text']}")
    print(f"  True label:  {label_map[row['true_label']]}")
    print(f"  Predicted:   {label_map[row['predicted_label']]}")
    print(f"  Explanation: {row['explanation']}")
    print()

---
## Step 5: Build Few-Shot Prompt

In [ ]:
def build_few_shot_prompt(df_4_examples, test_text):
    """
    Builds a few-shot prompt using 4 labelled + explained examples.
    The full prompt is passed to DeBERTa as a single input sequence.
    Note: DeBERTa is an encoder model so the prompt acts as additional
    context rather than true generative few-shot learning.
    """
    prompt = "Task: Detect sarcasm. Examples:\n\n"

    for i, row in df_4_examples.iterrows():
        prompt += f"Text: \"{row['text']}\" "
        prompt += f"Label: {label_map[row['true_label']]}. "
        prompt += f"Reason: {row['explanation']}\n\n"

    prompt += f"Text: \"{test_text}\" Label:"
    return prompt

# Preview with first of the remaining 6
df_6 = df_10.drop(df_10.index[chosen_indices]).reset_index(drop=True)
print(build_few_shot_prompt(df_4, df_6.iloc[0]['text']))

---
## Step 6: Test Remaining 6 Examples with the Few-Shot Prompt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
deberta_model = trainer.model.to(device)
deberta_model.eval()

def predict_with_prompt(prompt):
    """Tokenise and run DeBERTa inference on a few-shot prompt string."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = deberta_model(**inputs)

    pred_id = torch.argmax(outputs.logits, dim=-1).item()
    return pred_id

results = []
for _, row in df_6.iterrows():
    prompt = build_few_shot_prompt(df_4, row['text'])
    few_shot_pred = predict_with_prompt(prompt)

    results.append({
        'text':             row['text'],
        'variety':          row['variety'],
        'true_label':       row['true_label'],
        'original_pred':    row['predicted_label'],
        'few_shot_pred':    few_shot_pred,
        'original_correct': row['predicted_label'] == row['true_label'],
        'few_shot_correct': few_shot_pred == row['true_label'],
    })

df_results = pd.DataFrame(results)
df_results[['text', 'variety', 'true_label', 'original_pred', 'few_shot_pred', 'few_shot_correct']]

---
## Step 7: Analysis

In [ ]:
original_correct = df_results['original_correct'].sum()
few_shot_correct = df_results['few_shot_correct'].sum()
total = len(df_results)

print("=" * 55)
print("RESULTS SUMMARY (6 test examples)")
print("=" * 55)
print(f"Original DeBERTa (zero-shot) correct:  {original_correct}/{total}")
print(f"Few-shot DeBERTa correct:              {few_shot_correct}/{total}")
print()

for _, row in df_results.iterrows():
    status = '✅ IMPROVED'  if (not row['original_correct'] and row['few_shot_correct']) else \
             '❌ REGRESSED' if (row['original_correct'] and not row['few_shot_correct']) else \
             '➡️  NO CHANGE'
    print(f"[{status}] ({row['variety']}) {row['text'][:65]}")
    print(f"   True: {label_map[row['true_label']]} | "
          f"Original: {label_map[row['original_pred']]} | "
          f"Few-shot: {label_map[row['few_shot_pred']]}")
    print()

In [ ]:
categories = ['Original DeBERTa\n(zero-shot)', 'DeBERTa\n(few-shot prompt)']
correct_counts = [original_correct, few_shot_correct]
wrong_counts   = [total - original_correct, total - few_shot_correct]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, correct_counts, width, label='Correct', color='#4CAF50')
bars2 = ax.bar(x + width/2, wrong_counts,   width, label='Wrong',   color='#F44336')

ax.set_ylabel('Number of Examples')
ax.set_title('DeBERTa: Zero-Shot vs Few-Shot Prompt\n(on 6 erroneous test examples)')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, total + 1)

for bar in list(bars1) + list(bars2):
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center')

plt.tight_layout()
plt.savefig('few_shot_comparison.png', dpi=150)
plt.show()
print("Chart saved as few_shot_comparison.png")

---
## Step 8: Written Commentary

> ⚠️ **TODO:** Fill in the bracketed sections once you have real results.

### Commentary on Few-Shot Prompt Effectiveness

**Overview of errors extracted:**  
We extracted 10 erroneous predictions from the best-performing DeBERTa-v3-base model trained on all varieties combined (Section 2.2). The errors consisted of [X] False Negatives (missed sarcasm) and [Y] False Positives (wrongly flagged as sarcastic), spanning en-UK, en-IN, and en-AU.

**Why the 4 examples were chosen:**  
The 4 examples were selected because they represent clearly explainable failure modes — cases where the sarcasm can be attributed to a specific linguistic device such as [e.g. exaggerated positivity / understatement / rhetorical questions / dialect-specific phrasing]. This made it possible to write concise, generalizable explanations for the prompt.

**Results on the remaining 6 examples:**  
The few-shot prompt correctly classified [X/6] examples, compared to [Y/6] by the original zero-shot model.
- Examples that **improved**: [list] — the in-context examples helped DeBERTa recognise sarcasm patterns it had previously missed.
- Examples that **did not improve**: [list] — these involved subtler or more dialect-specific sarcasm that the few-shot examples did not generalise to.

**Key observations:**  
- DeBERTa is a classification encoder, not a generative model, so the few-shot prompt does not function the same way it would for a generative LLM. Rather than guiding step-by-step reasoning, it prepends additional sarcasm-relevant context to the input sequence.
- Any improvement is therefore likely attributable to the additional vocabulary and sarcasm-relevant tokens in the prompt shifting the model's attention, rather than true in-context learning.
- Overall, the few-shot prompt [did / did not] provide a meaningful improvement, suggesting that [conclusion — e.g. encoder models benefit less from few-shot prompting than generative LLMs for nuanced tasks like cross-dialect sarcasm detection].